# Exploratory Data Analysis (EDA) & Statistical Insights

**Project:** Retail Sales EDA  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy  
**Input:** `clean_dataset.csv` created in the previous data-cleaning task.

## Objective
Explore retail sales patterns using descriptive statistics, distributions, correlation analysis, box plots and three exploratory business hypotheses.

> **Data limitation:** The supplied workbook contains only 12 raw rows. After removing one duplicate and rejecting invalid records, the cleaned file contains 6 rows. This is a tiny sample. Statistical tests below are demonstrations only; p-values and patterns must not be treated as reliable business conclusions. The assignment's 10,000+ row requirement needs a larger real dataset.

## 1. Import libraries and load the cleaned data

Upload `clean_dataset.csv` to the same Google Colab session as this notebook, or keep it in the same folder when using Jupyter locally.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme()
df = pd.read_csv("clean_dataset.csv")

# Convert date and ensure numeric columns have numeric types
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
numeric_cols = [
    "quantity", "unit_price", "discount_pct",
    "gross_amount", "discount_amount", "net_revenue"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset shape:", df.shape)
display(df.head())

## 2. Data overview and quality check

Review data types, missing values, duplicate rows and categorical counts.

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("data_type"))

print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())

for col in ["customer_segment", "category", "city", "payment_status"]:
    print(f"\\n{col} counts:")
    display(df[col].value_counts(dropna=False).to_frame("count"))

## 3. Descriptive statistics

Calculate count, mean, median, standard deviation, minimum, quartiles and maximum for the numeric measures.

In [ ]:
summary = df[numeric_cols].describe().T
summary["median"] = df[numeric_cols].median()
summary = summary[[
    "count", "mean", "median", "std", "min", "25%", "50%", "75%", "max"
]]
display(summary.round(2))

## 4. Distribution analysis — histograms

Histograms show how quantities, discounts and net revenue are distributed. With only six rows, each bar represents very few observations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(
    axes,
    ["quantity", "discount_pct", "net_revenue"],
    ["Quantity distribution", "Discount percentage distribution", "Net revenue distribution"]
):
    sns.histplot(data=df, x=col, bins=5, kde=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel(col.replace("_", " ").title())
plt.tight_layout()
plt.show()

## 5. Box plots — potential outliers

Box plots display the median, quartiles and potential outliers. A flagged point is not automatically an error; it should be checked against business context.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["quantity", "discount_pct", "net_revenue"]):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(f"Box plot: {col.replace('_', ' ').title()}")
plt.tight_layout()
plt.show()

## 6. Correlation analysis

Pearson correlation measures linear association between numeric variables. Correlation does not prove causation. Because this dataset is very small, correlations can change sharply when one record is added or removed.

In [ ]:
corr = df[numeric_cols].corr(method="pearson")
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation heatmap of numeric variables")
plt.tight_layout()
plt.show()

display(corr.round(2))

## 7. Business hypotheses and statistical tests

We define three questions before looking at test results. Spearman correlation is used for monotonic relationships; Mann–Whitney U compares two independent groups. These tests are exploratory only because the sample is extremely small.

### Hypothesis 1 — Discount rate and net revenue
- **H0:** There is no monotonic association between discount percentage and net revenue.
- **H1:** There is a monotonic association between discount percentage and net revenue.
- **Test:** Spearman rank correlation.

In [ ]:
h1 = df[["discount_pct", "net_revenue"]].dropna()
if len(h1) >= 3 and h1["discount_pct"].nunique() > 1 and h1["net_revenue"].nunique() > 1:
    r1, p1 = stats.spearmanr(h1["discount_pct"], h1["net_revenue"])
    print(f"Spearman rho = {r1:.3f}, p-value = {p1:.3f}, n = {len(h1)}")
else:
    print("Not enough variation or observations to run this test.")

### Hypothesis 2 — Quantity and net revenue
- **H0:** There is no monotonic association between quantity and net revenue.
- **H1:** There is a monotonic association between quantity and net revenue.
- **Test:** Spearman rank correlation.

In [ ]:
h2 = df[["quantity", "net_revenue"]].dropna()
if len(h2) >= 3 and h2["quantity"].nunique() > 1 and h2["net_revenue"].nunique() > 1:
    r2, p2 = stats.spearmanr(h2["quantity"], h2["net_revenue"])
    print(f"Spearman rho = {r2:.3f}, p-value = {p2:.3f}, n = {len(h2)}")
else:
    print("Not enough variation or observations to run this test.")

### Hypothesis 3 — Paid versus non-paid order revenue
- **H0:** The net revenue distributions are the same for Paid and non-Paid orders.
- **H1:** The net revenue distributions differ between Paid and non-Paid orders.
- **Test:** Mann–Whitney U test.
- **Caution:** Revenue on pending/refunded orders is not necessarily realized cash. This comparison is descriptive and should not be used as a financial performance conclusion.

In [ ]:
paid = df.loc[df["payment_status"].str.casefold().eq("paid"), "net_revenue"].dropna()
non_paid = df.loc[~df["payment_status"].str.casefold().eq("paid"), "net_revenue"].dropna()

print("Paid group size:", len(paid))
print("Non-paid group size:", len(non_paid))

if len(paid) >= 2 and len(non_paid) >= 2:
    u_stat, p3 = stats.mannwhitneyu(paid, non_paid, alternative="two-sided")
    print(f"Mann–Whitney U = {u_stat:.3f}, p-value = {p3:.3f}")
else:
    print("Insufficient observations in one or both groups for a meaningful comparison.")

## 8. Top five findings from this sample

The following findings are calculated from the six valid sample rows. They are sample observations, not general retail-market conclusions.

In [ ]:
print(f"1. Valid orders analysed: {df['order_id'].nunique()}")
print(f"2. Total quantity: {df['quantity'].sum():.0f}")
print(f"3. Average net revenue per row: {df['net_revenue'].mean():.2f}")
print(f"4. Median net revenue per row: {df['net_revenue'].median():.2f}")
print(f"5. Orders marked Paid: {(df['payment_status'].str.casefold() == 'paid').sum()} out of {len(df)} rows")

print("\\nNet revenue by customer segment:")
display(df.groupby("customer_segment")["net_revenue"].agg(["count", "mean", "median"]).round(2))

print("\\nNet revenue by payment status:")
display(df.groupby("payment_status")["net_revenue"].agg(["count", "sum", "mean"]).round(2))

## 9. Interpretation and recommendations

1. Use the six-row output only to demonstrate the EDA workflow; do not generalize its patterns.
2. Obtain at least 10,000 genuine retail transaction rows to meet the task specification and improve statistical power.
3. Confirm whether missing discount values truly mean 0% before using that imputation rule.
4. Add cost, customer ID and order history fields if the business wants profit margin, retention and repeat-purchase analysis.
5. Interpret payment status carefully: pending, failed and refunded transactions should not automatically be treated as realized revenue.

## 10. Conclusion

This notebook demonstrates descriptive statistics, histograms, box plots, correlation heatmaps and three hypothesis tests using the available cleaned retail sample. A larger, representative dataset is required before drawing statistically reliable business insights.